# Treino, avaliacao e inferencia - deteccao de helipontos

Projeto P2 (CDIA - PUC-SP, Prof. Dr. Rooney Coelho). Notebook de treino/avaliacao/inferencia.
Modelo principal YOLO26n (Ultralytics), com YOLOv8n/YOLO11n como fallback. Roda no Google Colab (GPU T4).

Pipeline: baixar o dataset versionado do Roboflow, sanity checks, treinar 2 rodadas variando 1
hiperparametro, avaliar no teste (Avenida Paulista, bairro inedito), analisar erros 5/3/3 e fazer inferencia.

Atribuicao obrigatoria: Source: Esri, Maxar, Earthstar Geographics, and the GIS User Community

Referencias de aula: notebook 03_yolo_e_metricas (IoU, NMS, mAP), notebook 05_formatos_e_ferramentas
(formato YOLO), notas de CNN PyTorch (reprodutibilidade, transfer learning) e a sintese CDIA 2026.

Seguranca: a chave de API do Roboflow nao fica escrita no codigo. Ela e pedida na hora com getpass.
Se a sua chave ja foi exposta em algum lugar, gere uma nova no Roboflow.

In [ ]:
# no colab descomente para instalar as dependencias
# !pip install -q ultralytics roboflow

import os
import random
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

# pasta base do projeto no drive (monte o drive antes no colab)
# from google.colab import drive
# drive.mount('/content/drive')
PASTA_BASE = '/content/drive/MyDrive/yolo_satelite_puc_sp'
ATRIBUICAO = 'Source: Esri, Maxar, Earthstar Geographics, and the GIS User Community'


def fixar_seed(seed=42):
    # fixa as sementes para tornar a execucao reproduzivel, conforme aula
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


fixar_seed(42)

# objeto-alvo e justificativa da escolha
OBJETO_ALVO = 'heliponto'
print('objeto-alvo:', OBJETO_ALVO)
print('justificativa: sao paulo tem a maior frota de helicopteros urbanos do mundo. a marcacao de heliponto')
print('(um H branco dentro de um circulo, pintado em coberturas) e um alvo visualmente distinto e quase')
print('exclusivo da cidade. e abundante na regiao da puc-sp e tem forma geometrica clara em vista de topo,')
print('o que facilita a anotacao e o aprendizado do detector a 0,27 metros por pixel (zoom 19).')

## 1. Baixar a versao fixa do dataset no Roboflow

A chave de API e lida do ambiente ou pedida com getpass. Ela nao deve ser escrita no codigo nem commitada.

In [ ]:
from getpass import getpass
from roboflow import Roboflow


def baixar_dataset_roboflow():
    # le a chave do ambiente ou pede de forma interativa, sem deixar a chave no codigo
    chave = os.environ.get('ROBOFLOW_API_KEY')
    if chave is None:
        chave = getpass('informe a chave de api do roboflow: ')
    # baixa a versao fixa 1 no formato yolo26, garantindo reprodutibilidade
    rf = Roboflow(api_key=chave)
    projeto = rf.workspace('yolo-maps-pucsp-vie').project('yolov26-maps-vie')
    versao = projeto.version(1)
    dataset = versao.download('yolo26')
    return dataset


dataset = baixar_dataset_roboflow()
caminho_data = os.path.join(dataset.location, 'data.yaml')
print('dataset baixado em:', dataset.location)
print('data.yaml:', caminho_data)

## 2. Sanity checks antes do treino

Confere classe unica, rotulos normalizados, ausencia de augmentation em val/test e as contagens.

In [ ]:
import yaml


def validar_yaml(caminho):
    # aceita apenas nc=1 e names contendo heliponto, conforme requirement 1.5 e 1.7
    with open(caminho, 'r') as arquivo:
        config = yaml.safe_load(arquivo)
    nc = config.get('nc')
    nomes = config.get('names')
    print('nc:', nc, '| names:', nomes)
    if nc != 1:
        print('erro: o dataset deve ter exatamente 1 classe, encontrado nc =', nc)
        return False
    return True


def rotulo_valido(cx, cy, w, h):
    # rejeita anotacoes fora do intervalo [0,1]
    valores = [cx, cy, w, h]
    for valor in valores:
        if valor < 0.0 or valor > 1.0:
            return False
    if cx - w / 2.0 < 0.0 or cx + w / 2.0 > 1.0:
        return False
    if cy - h / 2.0 < 0.0 or cy + h / 2.0 > 1.0:
        return False
    return True


def conferir_rotulos(pasta_labels):
    # percorre os arquivos de rotulo e confere se todas as caixas estao normalizadas
    total = 0
    invalidos = 0
    if not os.path.isdir(pasta_labels):
        return total, invalidos
    for nome in os.listdir(pasta_labels):
        if not nome.endswith('.txt'):
            continue
        with open(os.path.join(pasta_labels, nome), 'r') as arquivo:
            linhas = arquivo.readlines()
        for linha in linhas:
            partes = linha.split()
            if len(partes) < 5:
                continue
            total = total + 1
            if not rotulo_valido(float(partes[1]), float(partes[2]), float(partes[3]), float(partes[4])):
                invalidos = invalidos + 1
    return total, invalidos


def contar_imagens(pasta_images):
    # conta arquivos de imagem em uma pasta
    if not os.path.isdir(pasta_images):
        return 0
    total = 0
    for nome in os.listdir(pasta_images):
        if nome.endswith('.jpg') or nome.endswith('.png') or nome.endswith('.jpeg'):
            total = total + 1
    return total


ok_yaml = validar_yaml(caminho_data)
for particao in ['train', 'valid', 'test']:
    pasta_labels = os.path.join(dataset.location, particao, 'labels')
    pasta_images = os.path.join(dataset.location, particao, 'images')
    total_rotulos, invalidos = conferir_rotulos(pasta_labels)
    n_imagens = contar_imagens(pasta_images)
    print(particao, '-> imagens:', n_imagens, '| rotulos:', total_rotulos, '| invalidos:', invalidos)
print('validacao do yaml:', ok_yaml)
print('lembrete: augmentation deve estar somente no treino; valid e test sem augmentation')

## 3. Treino com transfer learning (2 rodadas variando 1 hiperparametro)

Modelo principal YOLO26n. Para usar o fallback, troque os pesos por yolov8n.pt ou yolo11n.pt.

In [ ]:
from ultralytics import YOLO

PESOS_BASE = 'yolo26n.pt'
# fallback caso o yolo26n nao esteja disponivel no ambiente:
# PESOS_BASE = 'yolo11n.pt'
# PESOS_BASE = 'yolov8n.pt'

PASTA_RUNS = os.path.join(PASTA_BASE, 'runs', 'detect')


def treinar(nome_rodada, epocas, batch, seed=42):
    # treina o modelo a partir de pesos pre-treinados (transfer learning)
    fixar_seed(seed)
    modelo = YOLO(PESOS_BASE)
    resultado = modelo.train(
        data=caminho_data,
        epochs=epocas,
        imgsz=640,
        batch=batch,
        device=0,
        seed=seed,
        project=PASTA_RUNS,
        name=nome_rodada,
    )
    print('rodada', nome_rodada, '| epochs', epocas, '| batch', batch, '| seed', seed)
    return modelo


# rodada 1 e rodada 2 variam apenas o numero de epocas; o resto fica identico
modelo_r1 = treinar('rodada1', epocas=30, batch=16, seed=42)
modelo_r2 = treinar('rodada2', epocas=50, batch=16, seed=42)

In [ ]:
# compara as rodadas pela mAP@0.5:0.95 de validacao e escolhe a melhor
def metrica_validacao(modelo):
    metricas = modelo.val(data=caminho_data, split='val', conf=0.25, iou=0.50)
    return float(metricas.box.map)


map_r1 = metrica_validacao(modelo_r1)
map_r2 = metrica_validacao(modelo_r2)
print('rodada1 (30 epocas) mAP@0.5:0.95 val:', round(map_r1, 4))
print('rodada2 (50 epocas) mAP@0.5:0.95 val:', round(map_r2, 4))

if map_r2 >= map_r1:
    melhor_rodada = 'rodada2'
    modelo_melhor = modelo_r2
else:
    melhor_rodada = 'rodada1'
    modelo_melhor = modelo_r1

caminho_best = os.path.join(PASTA_RUNS, melhor_rodada, 'weights', 'best.pt')
print('melhor rodada:', melhor_rodada)
print('best.pt:', caminho_best)

## 4. Avaliacao no conjunto de teste (Avenida Paulista)

Reporta mAP@0.5, mAP@0.5:0.95, precisao e revocacao e gera a matriz de confusao.

In [ ]:
modelo = YOLO(caminho_best)
metricas_teste = modelo.val(data=caminho_data, split='test', conf=0.25, iou=0.50)
print('mAP@0.5:     ', round(float(metricas_teste.box.map50), 4))
print('mAP@0.5:0.95:', round(float(metricas_teste.box.map), 4))
print('precisao:    ', round(float(metricas_teste.box.mp), 4))
print('revocacao:   ', round(float(metricas_teste.box.mr), 4))
print('a matriz de confusao e salva automaticamente na pasta da avaliacao (confusion_matrix.png)')
print('observacao: a paulista foi varrida inteira, entao a maioria dos tiles e fundo sem heliponto,')
print('o que aumenta a variancia da mAP de teste. a validacao tem apenas 20 imagens, tambem com mais variancia.')

## 5. Analise de erros (5 acertos, 3 falsos positivos, 3 falsos negativos)

Casa as deteccoes (conf >= 0.25) com as anotacoes verdadeiras por IoU >= 0.50 e coleta exemplos.

In [ ]:
def matriz_iou(A, B):
    # iou entre N caixas (A) e M caixas (B) em formato xyxy, via broadcasting numpy
    A = np.asarray(A, dtype=float)
    B = np.asarray(B, dtype=float)
    if A.size == 0 or B.size == 0:
        return np.zeros((A.shape[0], B.shape[0]), dtype=float)
    x1 = np.maximum(A[:, None, 0], B[None, :, 0])
    y1 = np.maximum(A[:, None, 1], B[None, :, 1])
    x2 = np.minimum(A[:, None, 2], B[None, :, 2])
    y2 = np.minimum(A[:, None, 3], B[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area_a = (A[:, 2] - A[:, 0]) * (A[:, 3] - A[:, 1])
    area_b = (B[:, 2] - B[:, 0]) * (B[:, 3] - B[:, 1])
    uniao = area_a[:, None] + area_b[None, :] - inter
    return np.where(uniao > 0, inter / np.maximum(uniao, 1e-12), 0.0)


def carregar_gt(caminho_label, largura, altura):
    # le o rotulo yolo (cxcywh normalizado) e devolve caixas xyxy em pixels
    caixas = []
    if not os.path.exists(caminho_label):
        return caixas
    with open(caminho_label, 'r') as arquivo:
        linhas = arquivo.readlines()
    for linha in linhas:
        partes = linha.split()
        if len(partes) < 5:
            continue
        cx = float(partes[1])
        cy = float(partes[2])
        w = float(partes[3])
        h = float(partes[4])
        x1 = (cx - w / 2.0) * largura
        y1 = (cy - h / 2.0) * altura
        x2 = (cx + w / 2.0) * largura
        y2 = (cy + h / 2.0) * altura
        caixas.append([x1, y1, x2, y2])
    return caixas


def contar_tp_fp_fn(gt, pred, limiar=0.5):
    # casa cada gt com a melhor pred disponivel e conta tp, fp, fn
    if len(gt) == 0:
        return 0, len(pred), 0
    if len(pred) == 0:
        return 0, 0, len(gt)
    ious = matriz_iou(gt, pred)
    pred_casado = set()
    tp = 0
    for i in range(len(gt)):
        melhor_j = -1
        melhor_iou = limiar
        for j in range(len(pred)):
            if j in pred_casado:
                continue
            if ious[i, j] >= melhor_iou:
                melhor_iou = ious[i, j]
                melhor_j = j
        if melhor_j >= 0:
            tp = tp + 1
            pred_casado.add(melhor_j)
    fp = len(pred) - len(pred_casado)
    fn = len(gt) - tp
    return tp, fp, fn

In [ ]:
# percorre o teste, classifica cada imagem e coleta exemplos de acerto, fp e fn
pasta_test_images = os.path.join(dataset.location, 'test', 'images')
pasta_test_labels = os.path.join(dataset.location, 'test', 'labels')

exemplos_acerto = []
exemplos_fp = []
exemplos_fn = []

nomes_test = []
for nome in os.listdir(pasta_test_images):
    if nome.endswith('.jpg') or nome.endswith('.png') or nome.endswith('.jpeg'):
        nomes_test.append(nome)
nomes_test.sort()

for nome in nomes_test:
    caminho_img = os.path.join(pasta_test_images, nome)
    base = os.path.splitext(nome)[0]
    caminho_label = os.path.join(pasta_test_labels, base + '.txt')
    imagem = Image.open(caminho_img)
    largura, altura = imagem.size
    gt = carregar_gt(caminho_label, largura, altura)
    resultado = modelo.predict(source=caminho_img, conf=0.25, iou=0.50, verbose=False)
    caixas_pred = resultado[0].boxes.xyxy.cpu().numpy()
    pred = []
    for k in range(len(caixas_pred)):
        pred.append([float(caixas_pred[k][0]), float(caixas_pred[k][1]), float(caixas_pred[k][2]), float(caixas_pred[k][3])])
    tp, fp, fn = contar_tp_fp_fn(gt, pred)
    registro = {'nome': nome, 'caminho': caminho_img, 'gt': gt, 'pred': pred}
    if tp > 0 and len(exemplos_acerto) < 5:
        exemplos_acerto.append(registro)
    if fp > 0 and len(exemplos_fp) < 3:
        exemplos_fp.append(registro)
    if fn > 0 and len(exemplos_fn) < 3:
        exemplos_fn.append(registro)

print('acertos coletados:', len(exemplos_acerto), 'de 5')
print('falsos positivos coletados:', len(exemplos_fp), 'de 3')
print('falsos negativos coletados:', len(exemplos_fn), 'de 3')
if len(exemplos_acerto) < 5 or len(exemplos_fp) < 3 or len(exemplos_fn) < 3:
    print('aviso: nao havia exemplos suficientes no teste para todas as categorias; mostrando os disponiveis')

In [ ]:
def desenhar_exemplos(lista, titulo):
    # desenha as caixas verdadeiras (verde) e as detectadas (vermelho) lado a lado
    if len(lista) == 0:
        print('sem exemplos para', titulo)
        return
    fig, eixos = plt.subplots(1, len(lista), figsize=(5 * len(lista), 5))
    if len(lista) == 1:
        eixos = [eixos]
    for ax, registro in zip(eixos, lista):
        imagem = Image.open(registro['caminho'])
        ax.imshow(imagem)
        for caixa in registro['gt']:
            x1, y1, x2, y2 = caixa
            ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor='lime', linewidth=2))
        for caixa in registro['pred']:
            x1, y1, x2, y2 = caixa
            ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor='red', linewidth=2))
        ax.set_title(registro['nome'], fontsize=8)
        ax.axis('off')
    fig.suptitle(titulo + ' (verde = verdadeiro, vermelho = detectado)')
    plt.tight_layout()
    plt.show()


desenhar_exemplos(exemplos_acerto, 'acertos')
desenhar_exemplos(exemplos_fp, 'falsos positivos')
desenhar_exemplos(exemplos_fn, 'falsos negativos')
print('discussao: escreva no relatorio as causas provaveis de cada erro (ex.: marcas circulares parecidas')
print('com heliponto geram falso positivo; helipontos com sombra ou baixa nitidez geram falso negativo).')

## 6. Inferencia no bairro inedito (Avenida Paulista)

Roda a inferencia nos tiles da Paulista, salva as imagens com as caixas e conta deteccoes por tile.

In [ ]:
pasta_saida_inferencia = os.path.join(PASTA_BASE, 'imagens_ineditas')
os.makedirs(pasta_saida_inferencia, exist_ok=True)

# seleciona ate 50 tiles da paulista para a inferencia de generalizacao
tiles_paulista = []
for nome in nomes_test:
    tiles_paulista.append(os.path.join(pasta_test_images, nome))
tiles_paulista = tiles_paulista[0:50]

contador = 0
for caminho_tile in tiles_paulista:
    try:
        resultado = modelo.predict(source=caminho_tile, conf=0.25, save=True,
                                   project=PASTA_BASE, name='imagens_ineditas', exist_ok=True, verbose=False)
        n_caixas = len(resultado[0].boxes)
        if contador < 5:
            print(os.path.basename(caminho_tile), '-> deteccoes:', n_caixas)
        contador = contador + 1
    except Exception as erro:
        print('falha na inferencia do tile', os.path.basename(caminho_tile), '-', erro)

print('inferencia concluida em', contador, 'tiles da paulista. resultados em', pasta_saida_inferencia)
print('para o relatorio: discuta pelo menos 5 tiles (acertos, falsos positivos e ausencias).')
print(ATRIBUICAO)

## 7. (Opcional) Comparacao YOLO26n x YOLO11n

Treina uma rodada com YOLO11n nas mesmas condicoes para comparar no relatorio.

In [ ]:
# opcional: comparacao de arquitetura sob os mesmos hiperparametros da melhor rodada
def treinar_comparacao(pesos, nome_rodada, epocas=50, batch=16, seed=42):
    fixar_seed(seed)
    modelo_cmp = YOLO(pesos)
    modelo_cmp.train(data=caminho_data, epochs=epocas, imgsz=640, batch=batch,
                     device=0, seed=seed, project=PASTA_RUNS, name=nome_rodada)
    metricas = modelo_cmp.val(data=caminho_data, split='test', conf=0.25, iou=0.50)
    return float(metricas.box.map50), float(metricas.box.map)


# descomente para rodar a comparacao
# map50_11, map_11 = treinar_comparacao('yolo11n.pt', 'comparacao_yolo11n')
# print('yolo11n teste mAP@0.5:', round(map50_11, 4), '| mAP@0.5:0.95:', round(map_11, 4))